# 1、获取大模型

# 2、使用提示词模板

In [1]:
# 确保 Windows/Japanese locale 终端也能正常打印中文输出
import sys
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

#导入 dotenv 库的 load_dotenv 函数，用于加载环境变量文件（.env）中的配置
import dotenv
from langchain_openai import ChatOpenAI
import os

dotenv.load_dotenv()  #加载当前目录下的 .env 文件

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 创建大模型实例
llm = ChatOpenAI(model="qwen2.5-coder:1.5b")  # 默认使用 qwen2.5-coder:1.5b

# 直接提供问题，并调用llm
response = llm.invoke("什么是大模型？")
print(response.content)

大模型（Large Model，也称为超大规模模型、超大规模神经网络）是一种能够处理大量数据、能够进行复杂任务的人工智能模型。它通常由多个神经元组成，并且网络结构复杂，可以模拟人类大脑的某些功能，如语言理解和生成、图像识别、决策支持等。

大模型通过训练大量数据来学习和表达人类的决策思维和情感，从而能够对新的数据进行预测和分析。它们在处理信息、处理复杂问题、理解和生成高质量内容等方面具有优势，可以广泛应用于计算机视觉、自然语言处理、机器学习等多个领域。


In [2]:
from langchain_core.prompts import ChatPromptTemplate

# 需要注意的一点是，这里需要指明具体的role，在这里是system和用户
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是世界级的技术文档编写者"),
    ("user", "{input}")  # {input}为变量
])

# 我们可以把prompt和具体llm的调用和在一起。
chain = prompt | llm
message = chain.invoke({"input": "大模型中的LangChain是什么?"})
print(message.content)

# print(type(message))

LangChain 是一个开源的跨平台框架，用于简化大模型的数据管理和模型训练。它可以用于构建从数据处理、模型训练到模型评估和部署的整个模型生命周期，而不需要依赖于特定的框架或工具。它支持多种类型的模型，包括机器学习模型、自然语言处理模型等，以满足各种任务和需求。通过使用 LangChain，开发者可以更高效地利用大模型，加速创新和服务的快速部署。


# 3、 使用输出解析器

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser

# 初始化模型
llm = ChatOpenAI(model="qwen2.5-coder:1.5b")

# 创建提示模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是世界级的技术文档编写者。"),
    ("user", "{input}")
])

# 使用输出解析器
# output_parser = StrOutputParser()
output_parser = JsonOutputParser()

# 将其添加到上一个链中
# chain = prompt | llm
chain = prompt | llm | output_parser

# 调用它并提出同样的问题。答案是一个字符串，而不是ChatMessage
# chain.invoke({"input": "LangChain是什么?"})
print(chain.invoke({"input": "LangChain是什么? 用JSON格式回复，问题用question，回答用answer"}))

{'text': 'question: LangChain是什么?\nanswer: LangChain是一个开源、可扩展的语言模型技术栈。它由OpenAI的.ai团队开发，并且是一个生态系统，其中包括一系列的工具和预训练模型。通过这些工具，开发者可以轻松地构建和集成语言模型，以实现任务如摘要生成、对话机器人、翻译和语言理解等。'}


# 4、使用向量存储

In [3]:
# 导入和使用 WebBaseLoader
from langchain_community.document_loaders import WebBaseLoader
import bs4

loader = WebBaseLoader(
        web_path="https://www.gov.cn/xinwen/2020-06/01/content_5516649.htm",
        bs_kwargs=dict(parse_only=bs4.SoupStrainer(id="UCAP-CONTENT"))
    )
docs = loader.load()
# print(docs)

# 对于嵌入模型，这里通过 API调用
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")


from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 使用分割器分割文档
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
documents = text_splitter.split_documents(docs)
print(len(documents))
# 向量存储  embeddings 会将 documents 中的每个文本片段转换为向量，并将这些向量存储在 FAISS 向量数据库中
vector = FAISS.from_documents(documents, embeddings)

1


# 5、RAG(检索增强生成)

In [6]:
from langchain_core.prompts import PromptTemplate

retriever = vector.as_retriever()
retriever.search_kwargs = {"k": 3}
docs = retriever.invoke("建设用地使用权是什么？")

# for i,doc in enumerate(docs):
#     print(f"⭐第{i+1}条规定：")
#     print(doc)

# 6.定义提示词模版
prompt_template = """
你是一个问答机器人。
你的任务是根据下述给定的已知信息回答用户问题。
确保你的回复完全依据下述已知信息。不要编造答案。
如果下述已知信息不足以回答用户的问题，请直接回复"我无法回答您的问题"。

已知信息:
{info}

用户问：
{question}

请用中文回答用户问题。
"""
# 7.得到提示词模版对象
template = PromptTemplate.from_template(prompt_template)

# 8.得到提示词对象
prompt = template.format(info=docs, question='建设用地使用权是什么？')

## 9. 调用LLM
response = llm.invoke(prompt)
print(response.content)

建设用地使用权是指土地使用权人拥有或使用特定土地所有权的法律凭证，用于合法建造房屋、商业设施或其他用途，但不包括农村宅基地。


# 6、使用Agent

In [4]:
from langchain.tools.retriever import create_retriever_tool

# 检索器工具
retriever_tool = create_retriever_tool(
    retriever,
    "CivilCodeRetriever",
    "搜索有关中华人民共和国民法典的信息。关于中华人民共和国民法典的任何问题，您必须使用此工具!",
)

tools = [retriever_tool]

from langchain import hub
from langchain.agents import create_openai_functions_agent
from langchain.agents import AgentExecutor

# https://smith.langchain.com/hub
prompt = hub.pull("hwchase17/openai-functions-agent")

agent = create_openai_functions_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 运行代理
agent_executor.invoke({"input": "建设用地使用权是什么"})

NameError: name 'retriever' is not defined